# GridS-FiLM and matched comparison baseline

## 1. Load shared code
FiLM uses the instruction to predict scale and shift values. These modify pooled visual features before coordinate prediction.

In [1]:
from pathlib import Path
import sys, json
import torch, torchvision
project_dir = Path.home() / "t2-project"
sys.path.insert(0, str(project_dir))
from t2.train import train, load_policy
from t2.data import preparation, vocabulary, Demonstrations
manifest, normalization, manifest_hash = preparation(project_dir)
print("Project:", project_dir)

Project: /home/njmcmil/t2-project


## 2. Train GridS-FiLM
The sampling budget remains 16 tokens per camera. Compare against original GridS and uniform sampling using the same data, epoch budget, and action head.

In [2]:
film_run = train(project_dir, "grids_film", seed=0, epochs=10)
film_seed1_run = train(project_dir, "grids_film", seed=1, epochs=10)

Run: /home/njmcmil/t2-project/runs/grids_film_seed0_20261117T212845_384875Z
Epoch 1/10: train=0.97520, validation=1.00748
Epoch 2/10: train=0.87881, validation=0.88502
Epoch 3/10: train=0.75043, validation=0.74471
Epoch 4/10: train=0.68144, validation=0.67484
Epoch 5/10: train=0.59500, validation=0.62427
Epoch 6/10: train=0.54820, validation=0.62927
Epoch 7/10: train=0.50881, validation=0.52457
Epoch 8/10: train=0.48417, validation=0.53228
Epoch 9/10: train=0.47262, validation=0.51794
Epoch 10/10: train=0.46348, validation=0.55895
Run: /home/njmcmil/t2-project/runs/grids_film_seed1_20261117T212956_979682Z
Epoch 1/10: train=0.97414, validation=1.00304
Epoch 2/10: train=0.84663, validation=0.80218
Epoch 3/10: train=0.73283, validation=0.74552
Epoch 4/10: train=0.70075, validation=0.70044
Epoch 5/10: train=0.66173, validation=0.70456
Epoch 6/10: train=0.63512, validation=0.64982
Epoch 7/10: train=0.60429, validation=0.64379
Epoch 8/10: train=0.58252, validation=0.61574
Epoch 9/10: train=0

## 3. Train the matched vision-only control
Your existing notebook 02 pilot remains valid development work. Its pooling/head differs from the new shared comparison architecture. Train these fresh controls for fair final comparisons. They use the same dense token pipeline but receive a zero language vector.

In [3]:
vision_run = train(project_dir, "vision_only", seed=0, epochs=10)
vision_seed1_run = train(project_dir, "vision_only", seed=1, epochs=10)

Run: /home/njmcmil/t2-project/runs/vision_only_seed0_20261117T213106_238623Z
Epoch 1/10: train=1.00130, validation=1.02837
Epoch 2/10: train=0.92156, validation=0.88434
Epoch 3/10: train=0.80170, validation=0.78981
Epoch 4/10: train=0.76725, validation=0.76145
Epoch 5/10: train=0.74667, validation=0.76142
Epoch 6/10: train=0.72320, validation=0.74517
Epoch 7/10: train=0.68880, validation=0.75598
Epoch 8/10: train=0.65120, validation=0.69200
Epoch 9/10: train=0.62338, validation=0.66069
Epoch 10/10: train=0.60417, validation=0.61621
Run: /home/njmcmil/t2-project/runs/vision_only_seed1_20261117T213210_698169Z
Epoch 1/10: train=1.00157, validation=1.02832
Epoch 2/10: train=1.00104, validation=1.02733
Epoch 3/10: train=0.97982, validation=0.92854
Epoch 4/10: train=0.89024, validation=0.87079
Epoch 5/10: train=0.84790, validation=0.83277
Epoch 6/10: train=0.80001, validation=0.77980
Epoch 7/10: train=0.75786, validation=0.80476
Epoch 8/10: train=0.73362, validation=0.78405
Epoch 9/10: train

## 4. Review the chosen runs
These are validation results. Select final settings with your graduate teammate before looking at held-out rollout outcomes.

In [4]:
for directory in sorted((project_dir / "runs").iterdir()):
    summary = directory / "validation_summary.json"
    config = directory / "config.json"
    if summary.exists() and config.exists():
        settings = json.loads(config.read_text())
        if settings.get("architecture") == "compact_attention_v1":
            print(directory.name, summary.read_text())

dense_seed0_20261117T211545_858127Z {
  "selected_epoch": 9,
  "validation_mse": 0.619675495972236,
  "per_task": {
    "put_the_bowl_on_the_plate": 0.697488769836409,
    "put_the_bowl_on_the_stove": 0.5439941026334179
  },
  "checkpoint_reload_verified": true
}
dense_seed1_20261117T211813_215453Z {
  "selected_epoch": 10,
  "validation_mse": 0.6421160671032138,
  "per_task": {
    "put_the_bowl_on_the_plate": 0.673760551408353,
    "put_the_bowl_on_the_stove": 0.6113385391176027
  },
  "checkpoint_reload_verified": true
}
grids_film_seed0_20261117T212845_384875Z {
  "selected_epoch": 9,
  "validation_mse": 0.517944386228919,
  "per_task": {
    "put_the_bowl_on_the_plate": 0.5507465212369783,
    "put_the_bowl_on_the_stove": 0.4860409371497441
  },
  "checkpoint_reload_verified": true
}
grids_film_seed1_20261117T212956_979682Z {
  "selected_epoch": 9,
  "validation_mse": 0.613483181844155,
  "per_task": {
    "put_the_bowl_on_the_plate": 0.6253833856210638,
    "put_the_bowl_on_the_s